In [3]:
"""
=========================================================
MODELOS — VALIDACIÓN POR PACIENTE (StratifiedGroupKFold)
=========================================================
Un único script para las TRES configuraciones:
  - ICBHI sola
  - Fraiwan sola
  - Fusión (ICBHI + Fraiwan)

La validación cruzada agrupa por PACIENTE REAL:
  - ICBHI:   "102_1b1_..."             -> ICBHI_102
  - Fraiwan: "BP100_...", "DP100_...", -> FRAIWAN_100
             "EP100_..."                  (los 3 filtros del
                                           Littmann son el MISMO
                                           paciente, no 3 distintos)

Esto elimina la fuga de información ventana/filtro-paciente.

CÓMO USARLO:
  Cambia CONFIG a "ICBHI", "FRAIWAN" o "FUSION".
  Si algún fold falla por pocas clases, baja N_SPLITS a 5.
=========================================================
"""

import re
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from collections import Counter

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier, BaggingClassifier, RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    cohen_kappa_score, balanced_accuracy_score
)
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek

# =========================================================
# >>> PARÁMETROS A ELEGIR <<<
# =========================================================

CONFIG   = "ICBHI"   # "ICBHI", "FRAIWAN" o "FUSION"
N_SPLITS = 10          # baja a 5 si algún fold falla por pocas clases

# =========================================================
# RUTAS
# =========================================================

path_icbhi   = r"C:\Users\josem\Desktop\tfg\features_enriquecidas_corregido.csv"
path_fraiwan = r"C:\Users\josem\Desktop\tfg\features_fraiwan_limpio.csv"

# =========================================================
# CARGA SEGÚN CONFIGURACIÓN
# =========================================================

def cargar(path):
    return pd.read_csv(path, sep=';', decimal=',')

if CONFIG == "ICBHI":
    df = cargar(path_icbhi)
elif CONFIG == "FRAIWAN":
    df = cargar(path_fraiwan)
elif CONFIG == "FUSION":
    df_i = cargar(path_icbhi)
    df_f = cargar(path_fraiwan)
    if list(df_i.columns) != list(df_f.columns):
        raise ValueError("Las columnas de ICBHI y Fraiwan no coinciden.")
    df = pd.concat([df_i, df_f], ignore_index=True)
else:
    raise ValueError("CONFIG debe ser 'ICBHI', 'FRAIWAN' o 'FUSION'")

print("=" * 55)
print(f"CONFIGURACIÓN: {CONFIG}   (N_SPLITS = {N_SPLITS})")
print("=" * 55)

# =========================================================
# LIMPIEZA DE FILAS SIN ARCHIVO
# =========================================================

df = df.dropna(subset=['archivo'])
df = df[df['archivo'].astype(str).str.strip() != '']

# =========================================================
# NORMALIZACIÓN DE ETIQUETAS
# =========================================================

df['diagnostico'] = (
    df['diagnostico']
    .astype(str).str.strip().str.upper()
    .replace({'HEALTHY': 'NORMAL', 'BRONCHIECTASIS': 'BRON'})
)

classes_to_keep = ['COPD', 'NORMAL', 'BRON', 'PNEUMONIA', 'ASTHMA', 'HEART FAILURE']
df = df[df['diagnostico'].isin(classes_to_keep)]

# =========================================================
# EXTRAER patient_id REAL
# =========================================================
# ICBHI:   empieza por número      -> ICBHI_<num>
# Fraiwan: empieza por letras+num  -> FRAIWAN_<num>
#          (BP/DP/EP = filtros del MISMO paciente)
# =========================================================

def extraer_patient_id(archivo):
    archivo = str(archivo)
    m = re.match(r'^([A-Za-z]+)(\d+)_', archivo)   # Fraiwan: BP100_, DP100_, EP100_
    if m:
        return f'FRAIWAN_{m.group(2)}'
    m2 = re.match(r'^(\d+)_', archivo)             # ICBHI: 102_
    if m2:
        return f'ICBHI_{m2.group(1)}'
    return 'DESCONOCIDO'

df['patient_id'] = df['archivo'].apply(extraer_patient_id)

n_desconocidos = (df['patient_id'] == 'DESCONOCIDO').sum()
if n_desconocidos > 0:
    print(f"\nAVISO: {n_desconocidos} ventanas con patient_id DESCONOCIDO.")
    print(df[df['patient_id'] == 'DESCONOCIDO']['archivo'].unique()[:10])

print("\nDistribución por ventanas:")
print(df['diagnostico'].value_counts())
print(f"\nPacientes únicos: {df['patient_id'].nunique()}")
print("\nPacientes por clase:")
print(df.groupby('diagnostico')['patient_id'].nunique())

# =========================================================
# PREPARAR X, y, groups
# =========================================================

meta_cols    = ['archivo', 'diagnostico', 'ventana', 'patient_id']
feature_cols = [c for c in df.columns if c not in meta_cols]

for col in feature_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df = df.dropna(subset=feature_cols)

X      = df[feature_cols].values
y      = df['diagnostico'].values
groups = df['patient_id'].values

print(f"\nFeatures: {len(feature_cols)}")
print(f"Total ventanas (tras dropna): {len(df)}")

# =========================================================
# EVALUACIÓN
# =========================================================

def evaluar(nombre, y_true, y_pred):
    acc     = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    kappa   = cohen_kappa_score(y_true, y_pred)
    labels  = sorted(set(y_true))
    cm      = confusion_matrix(y_true, y_pred, labels=labels)
    cm_df   = pd.DataFrame(cm, index=labels, columns=labels)

    print(f"\n{'='*55}")
    print(f"  {nombre}")
    print(f"{'='*55}")
    print(f"  Accuracy:          {acc:.4f}")
    print(f"  Balanced Accuracy: {bal_acc:.4f}")
    print(f"  Cohen's Kappa:     {kappa:.4f}")
    print(f"\n{classification_report(y_true, y_pred, zero_division=0)}")
    print("Matriz de confusión:")
    print(cm_df)
    return bal_acc

# =========================================================
# CV POR PACIENTE CON SMOTE DENTRO DEL FOLD
# =========================================================

def cross_val_grupos(nombre, modelo_fn, X, y, groups, n_splits):
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=42)
    scores = []
    all_true, all_pred = [], []

    for fold, (tr_idx, val_idx) in enumerate(cv.split(X, y, groups)):
        X_tr, X_val = X[tr_idx], X[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]

        sc = MinMaxScaler()
        X_tr  = sc.fit_transform(X_tr)
        X_val = sc.transform(X_val)

        counter     = Counter(y_tr)
        min_samples = min(counter.values())
        k_neighbors = min(5, min_samples - 1)

        if k_neighbors < 1:
            X_tr_res, y_tr_res = X_tr, y_tr
        else:
            smt = SMOTETomek(
                smote=SMOTE(k_neighbors=k_neighbors, random_state=42),
                random_state=42)
            X_tr_res, y_tr_res = smt.fit_resample(X_tr, y_tr)

        modelo = modelo_fn()
        modelo.fit(X_tr_res, y_tr_res)
        y_pred = modelo.predict(X_val)

        bal = balanced_accuracy_score(y_val, y_pred)
        scores.append(bal)
        all_true.extend(y_val)
        all_pred.extend(y_pred)

        print(f"    Fold {fold+1}: Balanced Acc = {bal:.4f}  "
              f"(val: {len(val_idx)} ventanas, {len(set(groups[val_idx]))} pacientes)")

    scores = np.array(scores)
    print(f"\n  Media:  {scores.mean():.4f}")
    print(f"  Std:    {scores.std():.4f}")

    evaluar(f"{nombre} — CV agregada", all_true, all_pred)
    return scores

# =========================================================
# MODELOS
# =========================================================

def make_adaboost():
    return AdaBoostClassifier(
        estimator=DecisionTreeClassifier(
            max_depth=4, min_samples_leaf=2,
            class_weight='balanced', random_state=42),
        n_estimators=200, learning_rate=0.5, random_state=42)

def make_bagging():
    return BaggingClassifier(
        estimator=DecisionTreeClassifier(
            max_depth=None, class_weight='balanced', random_state=42),
        n_estimators=200, random_state=42, n_jobs=-1)

def make_rf():
    return RandomForestClassifier(
        n_estimators=500, class_weight='balanced_subsample',
        max_features='sqrt', random_state=42, n_jobs=-1)

def make_svm():
    return SVC(kernel='rbf', C=10, gamma='scale',
               class_weight='balanced', random_state=42)

# =========================================================
# EJECUCIÓN
# =========================================================

for nombre, fn in [("ADABOOST", make_adaboost),
                   ("BAGGING", make_bagging),
                   ("RANDOM FOREST", make_rf),
                   ("SVM RBF", make_svm)]:
    print(f"\n{'='*40}")
    print(f"{nombre} — {N_SPLITS}-fold CV POR PACIENTE con SMOTE")
    print(f"{'='*40}")
    cross_val_grupos(nombre, fn, X, y, groups, N_SPLITS)

# =========================================================
# MODELO FINAL (sobre todo el dataset)
# =========================================================

print(f"\n{'='*40}")
print("MODELO FINAL (entrenado en todo el dataset)")
print(f"{'='*40}")

scaler_final = MinMaxScaler()
X_scaled_all = scaler_final.fit_transform(X)

counter_all = Counter(y)
k_final     = min(5, min(counter_all.values()) - 1)
if k_final < 1:
    X_final_res, y_final_res = X_scaled_all, y
else:
    smt_final = SMOTETomek(
        smote=SMOTE(k_neighbors=k_final, random_state=42),
        random_state=42)
    X_final_res, y_final_res = smt_final.fit_resample(X_scaled_all, y)

modelo_final = make_rf()
modelo_final.fit(X_final_res, y_final_res)

print(f"Distribución tras SMOTE: {Counter(y_final_res)}")

importances = pd.Series(modelo_final.feature_importances_, index=feature_cols)
top = importances.sort_values(ascending=False).head(15)
print("\nTop 15 features:")
for feat, imp in top.items():
    bar = '█' * int(imp * 200)
    print(f"  {feat:<30} {imp:.4f}  {bar}")

print(f"\n{'='*40}")
print(f"FIN — CONFIGURACIÓN {CONFIG}")
print(f"{'='*40}")

CONFIGURACIÓN: ICBHI   (N_SPLITS = 10)

Distribución por ventanas:
diagnostico
COPD         3387
PNEUMONIA     148
NORMAL        135
BRON          116
ASTHMA          4
Name: count, dtype: int64

Pacientes únicos: 110

Pacientes por clase:
diagnostico
ASTHMA        1
BRON         13
COPD         64
NORMAL       26
PNEUMONIA     6
Name: patient_id, dtype: int64

Features: 41
Total ventanas (tras dropna): 3789

ADABOOST — 10-fold CV POR PACIENTE con SMOTE
    Fold 1: Balanced Acc = 0.7466  (val: 305 ventanas, 9 pacientes)
    Fold 2: Balanced Acc = 0.7538  (val: 619 ventanas, 13 pacientes)
    Fold 3: Balanced Acc = 0.5958  (val: 359 ventanas, 10 pacientes)
    Fold 4: Balanced Acc = 0.9342  (val: 240 ventanas, 11 pacientes)
    Fold 5: Balanced Acc = 0.4472  (val: 369 ventanas, 12 pacientes)
    Fold 6: Balanced Acc = 0.9155  (val: 589 ventanas, 10 pacientes)
    Fold 7: Balanced Acc = 0.5736  (val: 498 ventanas, 11 pacientes)
    Fold 8: Balanced Acc = 0.5115  (val: 303 ventanas, 11 pa